# Convert FlyWire Drosophila connectome to libsonata-compatible circuit

Converts the FlyWire v783 connectome data (~138k neurons, ~15M synapses) into
SONATA-format HDF5 node/edge files with a standard circuit config.

**Source:** `../fly-brain/data/` (FlyWire v783 completeness CSV + connectivity parquet)

**Output:** `../../../obi-output/nest_fly_brain/circuit/`

### Neuron model mapping

The original Shiu et al. Brian2 model uses voltage-based synapses:

    dv/dt = (v_0 - v + g) / t_mbr    (membrane, g in mV)
    dg/dt = -g / tau                  (single-exponential synapse)

This maps to NEST `iaf_psc_exp` with `C_m = tau_m`, which makes the
voltage-based weight (mV) numerically equal to the current-based weight (pA).

| Shiu et al. | Value | NEST `iaf_psc_exp` param |
|-------------|-------|--------------------------|
| v_0 / v_rst | -52 mV | `E_L`, `V_reset` |
| v_th | -45 mV | `V_th` |
| t_mbr | 20 ms | `tau_m` |
| tau | 5 ms | `tau_syn_ex`, `tau_syn_in` |
| t_rfc | 2.2 ms | `t_ref` |
| — | 20 pF | `C_m` (= tau_m for mV↔pA equivalence) |

Edge `conductance` stores `Excitatory x Connectivity * w_syn` (w_syn = 0.275),
giving weights in pA-equivalent units ready for NEST. Delay is constant 1.8 ms.

In [ ]:
import json

import h5py
import numpy as np
import pandas as pd
from pathlib import Path

FLY_BRAIN_DIR = Path("/Users/james/Documents/obi/code/obi-main/fly-brain")
COMPLETENESS_CSV = FLY_BRAIN_DIR / "data" / "2025_Completeness_783.csv"
CONNECTIVITY_PARQUET = FLY_BRAIN_DIR / "data" / "2025_Connectivity_783.parquet"
OUTPUT_DIR = Path("../../../obi-output/circuit/nest_fly_brain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_comp = pd.read_csv(COMPLETENESS_CSV, index_col=0)
df_con = pd.read_parquet(CONNECTIVITY_PARQUET)

n_neurons = len(df_comp)
n_edges = len(df_con)
n_exc = (df_con["Excitatory"] == 1).sum()
n_inh = (df_con["Excitatory"] == -1).sum()

print(f"Neurons: {n_neurons:,}")
print(f"Synapses: {n_edges:,} ({n_exc:,} excitatory, {n_inh:,} inhibitory)")

### Create SONATA node and edge HDF5 files

In [ ]:
# --- Nodes ---
nodes_path = OUTPUT_DIR / "fly_nodes.h5"

NEST_MODEL = "iaf_psc_exp"
DYNAMICS_PARAMS = {
    "E_L": -52.0,        # resting potential (mV)
    "V_reset": -52.0,    # reset potential (mV)
    "V_th": -45.0,       # spike threshold (mV)
    "tau_m": 20.0,       # membrane time constant (ms)
    "tau_syn_ex": 5.0,   # excitatory synaptic time constant (ms)
    "tau_syn_in": 5.0,   # inhibitory synaptic time constant (ms)
    "t_ref": 2.2,        # refractory period (ms)
    "C_m": 20.0,         # membrane capacitance (pF) = tau_m for mV/pA equivalence
    "V_m": -52.0,        # initial membrane potential (mV)
    "I_e": 0.0,          # external DC current (pA)
}

with h5py.File(nodes_path, "w") as f:
    pop = f.create_group("nodes/fly")
    pop.create_dataset("node_id", data=np.arange(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_type_id", data=np.zeros(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_group_id", data=np.zeros(n_neurons, dtype=np.uint64))
    pop.create_dataset("node_group_index", data=np.arange(n_neurons, dtype=np.uint64))
    grp = pop.create_group("0")
    grp.create_dataset("flywire_id", data=df_comp.index.values.astype(np.int64))
    grp.create_dataset("model_template", data=NEST_MODEL)
    dp = grp.create_group("dynamics_params")
    for param_name, param_value in DYNAMICS_PARAMS.items():
        dp.create_dataset(param_name, data=param_value)

print(f"Nodes written to {nodes_path} ({nodes_path.stat().st_size / 1e6:.1f} MB)")
print(f"Model: {NEST_MODEL}")
for k, v in DYNAMICS_PARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# --- Edges (sorted by target for SONATA convention) ---
W_SYN = 0.275  # mV (Brian2) = pA (NEST with C_m = tau_m)

edges_path = OUTPUT_DIR / "fly_fly_edges.h5"

sort_idx = df_con["Postsynaptic_Index"].values.argsort(kind="mergesort")
src_ids = df_con["Presynaptic_Index"].values[sort_idx]
tgt_ids = df_con["Postsynaptic_Index"].values[sort_idx]
conductance = df_con["Excitatory x Connectivity"].values[sort_idx].astype(np.float64) * W_SYN

with h5py.File(edges_path, "w") as f:
    pop = f.create_group("edges/fly_to_fly")
    pop.create_dataset("source_node_id", data=src_ids.astype(np.uint64))
    pop.create_dataset("target_node_id", data=tgt_ids.astype(np.uint64))
    pop.create_dataset("edge_type_id", data=np.zeros(n_edges, dtype=np.uint32))
    pop.create_dataset("edge_group_id", data=np.zeros(n_edges, dtype=np.uint16))
    pop.create_dataset("edge_group_index", data=np.arange(n_edges, dtype=np.uint32))
    grp = pop.create_group("0")
    grp.create_dataset("conductance", data=conductance)
    grp.create_dataset("delay", data=np.full(n_edges, 1.8, dtype=np.float64))

print(f"Edges written to {edges_path} ({edges_path.stat().st_size / 1e6:.1f} MB)")
print(f"Conductance = Excitatory x Connectivity * {W_SYN}")
print(f"  range: [{conductance.min():.3f}, {conductance.max():.3f}] pA")

### Create circuit config and node sets

In [ ]:
node_sets = {"All": {"population": "fly"}}
node_sets_path = OUTPUT_DIR / "node_sets.json"
with node_sets_path.open("w", encoding="utf-8") as f:
    json.dump(node_sets, f, indent=2)

circuit_config = {
    "manifest": {"$BASE_DIR": "."},
    "node_sets_file": str(node_sets_path.absolute()),
    "networks": {
        "nodes": [
            {
                "nodes_file": str(nodes_path.absolute()),
                "populations": {"fly": {"type": "point_process"}},
            },
        ],
        "edges": [
            {
                "edges_file": str(edges_path.absolute()),
                "populations": {"fly_to_fly": {"type": "chemical"}},
            },
        ],
    },
}

circuit_config_path = OUTPUT_DIR / "circuit_config.json"
with circuit_config_path.open("w", encoding="utf-8") as f:
    json.dump(circuit_config, f, indent=2)

print(f"Circuit config: {circuit_config_path}")
print(f"Node sets: {node_sets_path}")

In [ ]:
# Verify
with h5py.File(nodes_path, "r") as f:
    n = f["nodes/fly/node_type_id"].shape[0]
    model = f["nodes/fly/0/model_template"][()].decode()
    params = {k: f[f"nodes/fly/0/dynamics_params/{k}"][()] for k in f["nodes/fly/0/dynamics_params"]}
    print(f"Node population 'fly': {n:,} neurons")
    print(f"  model_template: {model}")
    print(f"  dynamics_params: {params}")

with h5py.File(edges_path, "r") as f:
    e = f["edges/fly_to_fly/source_node_id"].shape[0]
    w = f["edges/fly_to_fly/0/conductance"]
    print(f"\nEdge population 'fly_to_fly': {e:,} edges")
    print(f"  conductance range: [{w[:].min():.3f}, {w[:].max():.3f}] pA")
    print(f"  delay: {f['edges/fly_to_fly/0/delay'][0]:.1f} ms (constant)")